In [ ]:
# Hücre 1: Streamlit ve ngrok kurulumu
!pip install -q streamlit pyngrok

In [ ]:
# Hücre 2: Ngrok bağlantısı için
from pyngrok import ngrok
# Eğer kendi ngrok token'ın varsa buraya yazabilirsin: ngrok.set_auth_token("TOKEN_BURAYA")
print(" Ngrok hazır.")

 Ngrok hazır.


In [ ]:
# Hücre 3: app.py dosyasını Drive'a kaydet
import os
from google.colab import drive

# Google Drive'ı bağla (eğer henüz bağlanmadıysa)
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Hedef dizini oluştur (yoksa otomatik oluşturulur)
target_dir = "/content/drive/MyDrive/SentinelAI"
os.makedirs(target_dir, exist_ok=True)

app_code = """
import streamlit as st
import pandas as pd

# Sayfa Ayarları
st.set_page_config(page_title="SentinelAI Dashboard", page_icon="🛡️", layout="wide")

# Başlık
st.title("🛡️ SentinelAI - AI-Powered Cybersecurity Analyst")
st.markdown("### SOC Analist Dashboard (Prototip)")

# Veriyi yükle
@st.cache_data
def load_data():
    BASE = "/content/drive/MyDrive/SentinelAI"
    return pd.read_parquet(f"{BASE}/data/processed/logs_classified.parquet")

df = load_data()

# Sidebar (Yan Menü)
st.sidebar.header("Filtreler")
threat_filter = st.sidebar.multiselect(
    "Tehdit Tipi Seçin:",
    options=df["threat_type"].unique(),
    default=["Brute Force", "Unknown Anomaly"]
)

# Filtrelenmiş veri
filtered_df = df[df["threat_type"].isin(threat_filter)]

# Üst Metrikler
col1, col2, col3, col4 = st.columns(4)
col1.metric("Toplam Log", len(df))
col2.metric("Tespit Edilen Anomali", len(df[df["threat_type"] != "Normal"]))
col3.metric("Kritik Alarmlar (Brute Force)", len(df[df["threat_type"] == "Brute Force"]))
col4.metric("Yanlış Pozitif Oranı", "0.6%")

st.divider()

# İki Sütunlu Yapı
col_left, col_right = st.columns([2, 1])

with col_left:
    st.subheader(" Aktif Alarmlar")
    if len(filtered_df) == 0:
        st.info("Seçilen kriterlere uygun alarm bulunamadı.")
    else:
        for idx, row in filtered_df.iterrows():
            with st.expander(f" {row['threat_type']} - IP: {row['ip']}"):
                st.write(f"**Zaman:** {row['timestamp']}")
                st.write(f"**Kullanıcı:** {row['user']}")
                st.write(f"**Deneme Sayısı:** {row['attempts']}")
                st.write(f"**MITRE:** {row['mitre_id']}")
                st.info(f"**Gerekçe:** {row['reason']}")

with col_right:
    st.subheader(" Tehdit Dağılımı")
    threat_counts = df["threat_type"].value_counts()
    st.bar_chart(threat_counts)

    st.subheader(" MITRE ID Dağılımı")
    mitre_counts = df[df["mitre_id"].notna()]["mitre_id"].value_counts()
    st.bar_chart(mitre_counts)
"""

with open(f"{target_dir}/app.py", "w") as f:
    f.write(app_code)

print(" app.py başarıyla oluşturuldu!")

 app.py başarıyla oluşturuldu!


In [ ]:
# Hücre 4: Streamlit'i arka planda başlat ve ngrok linkini al
import subprocess
import time
from pyngrok import ngrok, exception
import os

# Token almak için: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "3MKOhexuR1ScL0XeiTexCg3eGcL_3ueHwi1mGjgC4ku94gRzP"

if NGROK_TOKEN == "YOUR_NGROK_AUTHTOKEN":
    print(" HATA: Lütfen 'YOUR_NGROK_AUTHTOKEN' alanını kendi ngrok token'ınızla değiştirin!")
else:
    # Token ayarla (Hata durumunda bypass et)
    try:
        ngrok.set_auth_token(NGROK_TOKEN)
    except exception.PyngrokNgrokError as e:
        if "already set" in str(e):
            print("Ngrok token'ı zaten yapılandırılmış, devam ediliyor...")
        else:
            raise e

    # Streamlit'i arka planda başlat
    os.system("streamlit run /content/drive/MyDrive/SentinelAI/app.py &>/content/logs.txt &")
    time.sleep(5)  # Başlaması için bekle

    # Ngrok tüneli aç (port 8501)
    public_url = ngrok.connect(8501)
    print(f" Dashboard Linkiniz: {public_url}")
    print("Bu linke tıklayarak Dashboard'u görebilirsiniz.")

Ngrok token'ı zaten yapılandırılmış, devam ediliyor...
 Dashboard Linkiniz: NgrokTunnel: "https://frill-ripeness-kelp.ngrok-free.dev" -> "http://localhost:8501"
Bu linke tıklayarak Dashboard'u görebilirsiniz.
